# Task 1 — Building permits

Run cells from top to bottom in a fresh Python/Colab session. Dependencies: pandas >= 2, numpy, requests and plotly >= 5.5.

**Inputs:** `task1_building_permits_backup.zip` and `task3_results.zip` (or corresponding CSV files).

Saved raw snapshots are the reproducible inputs. Live API extraction code is included separately in the bundle for provenance and is not run by these notebooks. Expected totals describe the supplied snapshot; a different snapshot needs new validation and updated findings.

**Scope:** all five boroughs, issuance dates in 2023–2025. Building source: https://data.cityofnewyork.us/dataset/rbx6-tga4 . Street source: https://data.cityofnewyork.us/d/tqtj-sjs8 .


In [ ]:
# Load existing CSVs or ZIP backups locally; upload only when an input is missing.
# Only the requested CSV member is read. ZIP contents are not extracted to disk.
# Raw snapshots are reused; this workflow does not redownload live data.
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
from IPython.display import display
try:
    import google.colab
    BASE = Path('/content')
except ImportError:
    BASE = Path.cwd()
DATA_DIR = BASE / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
def load_saved_csv(filename, **kwargs):
    roots = [DATA_DIR, BASE]
    def locate():
        for folder in roots:
            direct = folder / filename
            if direct.exists():
                return pd.read_csv(direct, **kwargs)
        archives = sorted({p for folder in roots for p in folder.glob('*.zip')})
        for path in archives:
            try:
                with ZipFile(path) as archive:
                    members = [n for n in archive.namelist()
                               if Path(n).name == filename]
                    if len(members) == 1:
                        with archive.open(members[0]) as handle:
                            return pd.read_csv(handle, **kwargs)
            except zipfile.BadZipFile:
                continue
        return None
    import zipfile
    result = locate()
    if result is None:
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError(f'Place {filename} or its ZIP in {DATA_DIR}') from exc
        print(f'Upload {filename} or the ZIP containing it.')
        files.upload()
        result = locate()
    if result is None:
        raise FileNotFoundError(f'Uploaded files did not contain {filename}.')
    return result

In [ ]:
# This cell imports the libraries for data analysis, downloading data,
# and creating interactive charts. It also sets the NYC dataset API URL
# and creates a folder for our cleaned data and output files.

import pandas as pd
import numpy as np
import time

import plotly.express as px

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)

DATASET_ID = "rbx6-tga4"
API_URL = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.json"

OUTPUT_DIR = BASE / "task1_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

Setup complete.


In [ ]:
# Restore the original raw building snapshot without contacting the API.
df_raw = load_saved_csv('building_permits_raw_2023_2025.csv', dtype='string')
assert df_raw.shape == (484414, 20), 'Unexpected building snapshot.'
display(df_raw.head())

### Cleaning methodology and counting rules

**Source and scope**
We used DOB NOW: Build – Approved Permits:
https://data.cityofnewyork.us/dataset/rbx6-tga4

We selected records with issued dates from 1 January 2023 up to,
but excluding, 1 January 2026. We downloaded 484,414 records in batches,
matching the API count at extraction, and retained 20 relevant columns.
The original dataset contains 46 columns.

This period provides three complete calendar years shared with the
selected street construction dataset. The analysis covers permits
recorded in DOB NOW, rather than all construction activity in NYC.

**Formats and categories**
We standardized whitespace and borough capitalization, reducing ten
borough labels to five consistent names. Dates were converted to
datetime values, and costs and coordinates to numeric values.
All conversions succeeded. Identifiers remained text to preserve
their formatting.

**Duplicates and exclusions**
We removed 25 rows identical across the 20 selected columns.
This is deduplication at our selected analysis-column level.

We set aside 3,200 additional records because they were labelled
"No Work Permit", contained a "not yet issued" placeholder, or lacked
part of the chosen issuance key. These conditions overlap.
Excluded records were saved separately for audit and review.

The retained dataset contains 481,189 work-type detail rows.

**Counting unit**
Tracking numbers were not sufficient identifiers: some tracking
number/date combinations referred to unrelated permits.

We therefore defined an analytical issuance using:
work permit + sequence number + issued date + borough.

This produces 408,566 distinct eligible issuances.
It is an operational counting definition for this analysis, not a
claim that these fields are an officially guaranteed unique identifier.

Renewals remain separate issuance activity. Signed-off records are
retained because current status does not undo historical issuance.

An issuance may cover several work types. Overall counts include it
once; category counts include it once per associated work type.
Category counts overlap and must not be added to obtain overall totals.

**Missing values**
We retained 2,273 detail rows with missing coordinates because borough
and issuance date are available for our chosen integration.
We did not impute coordinates, identifiers or missing expiration dates.

**Outliers and inconsistencies**
The upper cost IQR threshold was $292,500; 71,482 detail rows exceeded it.
We retained and flagged these values because large projects can have
high costs and the values have not been independently verified.
Costs are not used in the permit-count analysis.

There were 33,202 zero-cost detail rows and no negative costs.
Zero costs were retained without assuming they represent free work.

All available coordinates passed a broad NYC-area bounding-box check.
This is a screening check, not validation against exact city boundaries.

We flagged 9 detail rows with approval after issuance and 250 with
expiration before issuance. They remain in issuance-count analysis,
but these inconsistent dates should not be used to calculate durations.

**Aggregation and validation**
We aggregated distinct issuances by borough, year and month.
All 180 expected borough-month groups contained records, and their
counts summed to 408,566.

A separate work-type summary supports category filtering.
These tables are prepared for integration with street permit data.
Post-integration cleaning must still be performed after the join.

In [ ]:
# This cell checks missing values, blank strings, data types and exact duplicates.
# It also examines boroughs, filing reasons and current permit statuses.
# We inspect these issues before cleaning so every decision is supported by evidence.
# Repeated permit numbers are investigated separately because renewals may be valid records.

quality_summary = pd.DataFrame({
    "data_type": df_raw.dtypes.astype(str),
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2),
    "blank_string_count": [
        df_raw[col].astype("string").str.strip().eq("").sum()
        for col in df_raw.columns
    ],
    "unique_values": df_raw.nunique()
}).sort_values("missing_percent", ascending=False)

print("DATA QUALITY SUMMARY")
display(quality_summary)

print("\nEXACT DUPLICATE ROWS")
print(f"{df_raw.duplicated().sum():,}")

for column in ["borough", "filing_reason", "permit_status", "work_type"]:
    print(f"\nVALUES IN: {column}")
    display(
        df_raw[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="record_count")
    )

print("\nCANDIDATE IDENTIFIER CHECKS")
for keys in [
    ["tracking_number"],
    ["work_permit"],
    ["work_permit", "sequence_number"]
]:
    complete = df_raw[keys].notna().all(axis=1)

    repeated = df_raw.loc[complete].duplicated(
        subset=keys,
        keep=False
    ).sum()

    print(
        f"{' + '.join(keys)}: "
        f"{(~complete).sum():,} rows with missing key values; "
        f"{repeated:,} rows involved in repeated keys"
    )

DATA QUALITY SUMMARY


,data_type,missing_count,missing_percent,blank_string_count,unique_values
latitude,object,2296,0.47,0,75156
longitude,object,2296,0.47,0,74779
expired_date,object,712,0.15,0,3040
zip_code,object,606,0.13,0,222
filing_reason,object,0,0.00,0,4
sequence_number,object,8,0.00,0,19
work_permit,object,6,0.00,0,315534
job_filing_number,object,6,0.00,0,302832
house_no,object,2,0.00,0,19602
street_name,object,0,0.00,0,6794



EXACT DUPLICATE ROWS
25

VALUES IN: borough


,borough,record_count
0,MANHATTAN,144300
1,BROOKLYN,103350
2,QUEENS,81174
3,BRONX,37923
4,Manhattan,34274
5,STATEN ISLAND,25689
6,Brooklyn,24715
7,Queens,18340
8,Bronx,10009
9,Staten Island,4640



VALUES IN: filing_reason


,filing_reason,record_count
0,Initial Permit,326480
1,Renewal Permit Without Changes,120942
2,Renewal Permit with Changes,33808
3,No Work Permit,3184



VALUES IN: permit_status


,permit_status,record_count
0,Signed-off,313774
1,Permit Issued,170640



VALUES IN: work_type


,work_type,record_count
0,General Construction,120072
1,Plumbing,70508
2,Mechanical Systems,47089
3,Sidewalk Shed,37726
4,Structural,35242
5,Construction Fence,29923
6,Sprinklers,27515
7,Supported Scaffold,18911
8,Solar,17829
9,Foundation,16336



CANDIDATE IDENTIFIER CHECKS
tracking_number: 0 rows with missing key values; 124,987 rows involved in repeated keys
work_permit: 6 rows with missing key values; 266,099 rows involved in repeated keys
work_permit + sequence_number: 14 rows with missing key values; 123,600 rows involved in repeated keys


In [ ]:
# This cell creates a cleaning copy while preserving the raw dataset.
# It standardizes text spacing and borough capitalization, converts dates and
# numeric measurements, and keeps identifiers such as ZIP codes as text.
# Conversion failures are reported so invalid values are not silently ignored.
# No records are removed at this stage.

df_clean = df_raw.copy(deep=True)

# Normalize text without changing identifiers into numbers.
for column in df_clean.columns:
    df_clean[column] = (
        df_clean[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

df_clean["borough"] = df_clean["borough"].str.upper()

conversion_checks = []

# Parse dates and record any non-missing values that fail conversion.
for column in ["approved_date", "issued_date", "expired_date"]:
    original = df_clean[column].copy()
    converted = pd.to_datetime(
        original,
        format="ISO8601",
        errors="coerce"
    )

    failed = original.notna() & converted.isna()

    conversion_checks.append({
        "column": column,
        "conversion_failures": int(failed.sum())
    })

    df_clean[column] = converted

# Convert measurements; identifiers remain strings.
for column in ["estimated_job_costs", "latitude", "longitude"]:
    original = df_clean[column].copy()
    numeric_text = original

    if column == "estimated_job_costs":
        numeric_text = original.str.replace(
            r"[$,]", "", regex=True
        )

    converted = pd.to_numeric(numeric_text, errors="coerce")
    failed = original.notna() & converted.isna()

    conversion_checks.append({
        "column": column,
        "conversion_failures": int(failed.sum())
    })

    df_clean[column] = converted

print("CONVERSION CHECKS")
display(pd.DataFrame(conversion_checks))

print("\nSTANDARDIZED BOROUGHS")
display(df_clean["borough"].value_counts(dropna=False))

valid_boroughs = {
    "MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"
}

print(
    "\nRows with missing or unexpected boroughs:",
    int((~df_clean["borough"].isin(valid_boroughs)).sum())
)

print("\nIssued date range:")
print(df_clean["issued_date"].min(), "to", df_clean["issued_date"].max())

print(f"\nRows retained: {len(df_clean):,}")

CONVERSION CHECKS


,column,conversion_failures
0,approved_date,0
1,issued_date,0
2,expired_date,0
3,estimated_job_costs,0
4,latitude,0
5,longitude,0



STANDARDIZED BOROUGHS


,count
borough,
MANHATTAN,178574
BROOKLYN,128065
QUEENS,99514
BRONX,47932
STATEN ISLAND,30329



Rows with missing or unexpected boroughs: 0

Issued date range:
2023-01-02 00:00:00 to 2025-12-31 00:00:00

Rows retained: 484,414


In [ ]:
# This cell investigates repeated permit identifiers before deduplication.
# Renewals or different issuance dates may represent valid activity.
# We compare candidate keys and inspect examples to decide what one counted
# record should represent. No rows are removed in this cell.

candidate_keys = [
    ["tracking_number"],
    ["work_permit", "sequence_number"],
    ["work_permit", "sequence_number", "issued_date"]
]

key_results = []

for keys in candidate_keys:
    complete = df_clean[keys].notna().all(axis=1)
    eligible = df_clean.loc[complete]

    key_results.append({
        "candidate_key": " + ".join(keys),
        "rows_missing_key": int((~complete).sum()),
        "rows_in_repeated_keys": int(
            eligible.duplicated(subset=keys, keep=False).sum()
        ),
        "extra_rows_beyond_unique_keys": int(
            eligible.duplicated(subset=keys).sum()
        )
    })

print("CANDIDATE KEY SUMMARY")
display(pd.DataFrame(key_results))

print("\nIDENTICAL ROWS ACROSS SELECTED COLUMNS")
print(f"{df_clean.duplicated().sum():,}")

# Select three repeated work permits and display their records together.
permit_counts = df_clean["work_permit"].value_counts()
example_permits = permit_counts[permit_counts > 1].head(3).index

print("\nEXAMPLES OF REPEATED WORK PERMITS")
display(
    df_clean.loc[
        df_clean["work_permit"].isin(example_permits),
        [
            "work_permit",
            "sequence_number",
            "tracking_number",
            "filing_reason",
            "issued_date",
            "approved_date",
            "permit_status",
            "borough",
            "work_type"
        ]
    ].sort_values(
        ["work_permit", "issued_date", "sequence_number"]
    )
)

print("\nRECORDS WITH MISSING PERMIT OR SEQUENCE")
display(
    df_clean.loc[
        df_clean[["work_permit", "sequence_number"]]
        .isna().any(axis=1),
        [
            "job_filing_number",
            "work_permit",
            "sequence_number",
            "tracking_number",
            "filing_reason",
            "issued_date",
            "borough",
            "work_type"
        ]
    ]
)

CANDIDATE KEY SUMMARY


,candidate_key,rows_missing_key,rows_in_repeated_keys,extra_rows_beyond_unique_keys
0,tracking_number,0,124987,73942
1,work_permit + sequence_number,14,123600,73178
2,work_permit + sequence_number + issued_date,14,123325,73008



IDENTICAL ROWS ACROSS SELECTED COLUMNS
25

EXAMPLES OF REPEATED WORK PERMITS


,work_permit,sequence_number,tracking_number,filing_reason,issued_date,approved_date,permit_status,borough,work_type
311237,B00485579-I1-GC-CX,3,203541288,Renewal Permit Without Changes,2023-09-14,2021-06-15,Permit Issued,BROOKLYN,Foundation
311284,B00485579-I1-GC-CX,3,203541288,Renewal Permit Without Changes,2023-09-14,2021-06-15,Permit Issued,BROOKLYN,General Construction
311403,B00485579-I1-GC-CX,3,203541288,Renewal Permit Without Changes,2023-09-14,2021-06-15,Permit Issued,BROOKLYN,Earth Work
311565,B00485579-I1-GC-CX,3,203541288,Renewal Permit Without Changes,2023-09-14,2021-06-15,Permit Issued,BROOKLYN,Mechanical Systems
311766,B00485579-I1-GC-CX,3,203541288,Renewal Permit Without Changes,2023-09-14,2021-06-15,Permit Issued,BROOKLYN,Structural
...,...,...,...,...,...,...,...,...,...
331047,X00525964-I1-GC-CX,10,747371920,Renewal Permit Without Changes,2025-02-21,2022-05-04,Permit Issued,BRONX,General Construction
330886,X00525964-I1-GC-CX,11,148228110,Renewal Permit Without Changes,2025-09-04,2022-05-04,Permit Issued,BRONX,General Construction
330958,X00525964-I1-GC-CX,11,148228110,Renewal Permit Without Changes,2025-09-04,2022-05-04,Permit Issued,BRONX,Mechanical Systems
330991,X00525964-I1-GC-CX,11,148228110,Renewal Permit Without Changes,2025-09-04,2022-05-04,Permit Issued,BRONX,Foundation



RECORDS WITH MISSING PERMIT OR SEQUENCE


,job_filing_number,work_permit,sequence_number,tracking_number,filing_reason,issued_date,borough,work_type
11614,Permit is no,Permit is not yet issued,<NA>,640939807,Initial Permit,2024-02-21,MANHATTAN,Suspended Scaffold
25571,Permit is no,Permit is not yet issued,<NA>,638399759,Initial Permit,2024-01-30,MANHATTAN,Suspended Scaffold
70463,Permit is no,Permit is not yet issued,<NA>,723763876,Initial Permit,2024-04-03,MANHATTAN,Suspended Scaffold
73317,Permit is no,Permit is not yet issued,<NA>,745905899,Initial Permit,2024-01-30,BRONX,Suspended Scaffold
178174,<NA>,<NA>,6,795276260,Renewal Permit Without Changes,2023-05-05,BRONX,Sidewalk Shed
207658,Permit is no,Permit is not yet issued,<NA>,878687192,Initial Permit,2024-05-17,QUEENS,Solar
208197,<NA>,<NA>,6,693901045,Renewal Permit Without Changes,2023-01-27,MANHATTAN,Sidewalk Shed
224216,<NA>,<NA>,5,332785355,Renewal Permit Without Changes,2023-01-20,MANHATTAN,Sidewalk Shed
302627,Permit is no,Permit is not yet issued,<NA>,125298443,Initial Permit,2024-02-27,MANHATTAN,Suspended Scaffold
307884,Permit is no,Permit is not yet issued,<NA>,985802387,Initial Permit,2024-01-26,BROOKLYN,Suspended Scaffold


In [ ]:
# This cell tests tracking number + issued date as a candidate issuance key.
# Multiple work types may belong to the same issuance, so row counts can inflate totals.
# We check for conflicting boroughs and permit numbers before adopting this key.
# We also count records marked "No Work Permit" or "not yet issued" for review.

EVENT_KEY = ["tracking_number", "issued_date"]

event_checks = (
    df_clean.groupby(EVENT_KEY, dropna=False)
    .agg(
        row_count=("work_type", "size"),
        borough_count=("borough", "nunique"),
        permit_count=("work_permit", "nunique"),
        sequence_count=("sequence_number", "nunique"),
        filing_reason_count=("filing_reason", "nunique"),
        work_type_count=("work_type", "nunique")
    )
    .reset_index()
)

conflict_mask = (
    (event_checks["borough_count"] > 1)
    | (event_checks["permit_count"] > 1)
    | (event_checks["sequence_count"] > 1)
    | (event_checks["filing_reason_count"] > 1)
)

print("CANDIDATE ISSUANCE KEY CHECK")
print(f"Distinct candidate issuances: {len(event_checks):,}")
print(
    "Candidate issuances covering multiple work types:",
    f"{(event_checks['work_type_count'] > 1).sum():,}"
)
print(
    "Candidate issuances with conflicting attributes:",
    f"{conflict_mask.sum():,}"
)

print("\nREPEATED ISSUANCE + WORK TYPE KEYS")
detail_key = EVENT_KEY + ["work_type"]
print(
    f"{df_clean.duplicated(subset=detail_key).sum():,} extra rows"
)

no_work_mask = (
    df_clean["filing_reason"]
    .str.casefold()
    .eq("no work permit")
    .fillna(False)
)

not_issued_mask = (
    df_clean["work_permit"]
    .str.contains("not yet issued", case=False, na=False)
)

print("\nRECORDS REQUIRING REVIEW")
print(f"'No Work Permit' rows: {no_work_mask.sum():,}")
print(f"'Not yet issued' placeholder rows: {not_issued_mask.sum():,}")

if conflict_mask.any():
    conflict_keys = event_checks.loc[conflict_mask, EVENT_KEY].head(5)

    print("\nEXAMPLES OF CONFLICTING KEYS")
    display(
        df_clean.merge(conflict_keys, on=EVENT_KEY, how="inner")[
            EVENT_KEY + [
                "work_permit", "sequence_number", "borough",
                "filing_reason", "work_type", "permit_status"
            ]
        ].sort_values(EVENT_KEY).head(40)
    )

CANDIDATE ISSUANCE KEY CHECK
Distinct candidate issuances: 411,415
Candidate issuances covering multiple work types: 50,301
Candidate issuances with conflicting attributes: 16

REPEATED ISSUANCE + WORK TYPE KEYS
30 extra rows

RECORDS REQUIRING REVIEW
'No Work Permit' rows: 3,184
'Not yet issued' placeholder rows: 10

EXAMPLES OF CONFLICTING KEYS


,tracking_number,issued_date,work_permit,sequence_number,borough,filing_reason,work_type,permit_status
6,254290557,2025-06-10,B01151456-S3-PL,1,BROOKLYN,Initial Permit,Plumbing,Permit Issued
7,254290557,2025-06-10,M01185922-S2-PL,1,MANHATTAN,Initial Permit,Plumbing,Signed-off
1,281543448,2023-06-23,Q00762507-I1-SF,2,QUEENS,Renewal Permit Without Changes,Supported Scaffold,Signed-off
2,281543448,2023-06-23,B00764144-S2-ST,2,BROOKLYN,Renewal Permit with Changes,Structural,Permit Issued
3,295491102,2025-04-28,M01208258-I1-GC,1,MANHATTAN,Initial Permit,General Construction,Signed-off
4,295491102,2025-04-28,B01212505-I1-GC,1,BROOKLYN,Initial Permit,General Construction,Signed-off
0,322510328,2025-04-23,Q01213217-I1-SL,1,QUEENS,Initial Permit,Solar,Permit Issued
10,322510328,2025-04-23,M01206858-S1-PL,1,MANHATTAN,Initial Permit,Plumbing,Signed-off
5,438002114,2024-05-15,X00743376-I1-PL,4,BRONX,Renewal Permit Without Changes,Plumbing,Signed-off
8,438002114,2024-05-15,S01009555-I1-GC-CX,1,STATEN ISLAND,Initial Permit,General Construction,Permit Issued


In [ ]:
# This cell removes rows identical across our selected columns and sets aside
# records that cannot reliably represent an issued work permit.
# We exclude "No Work Permit", "not yet issued" placeholders and incomplete
# issuance keys from the analysis, but preserve them in a review table.
# Renewals and signed-off permits remain; work-type detail is also preserved.

df_before_exclusions = df_clean.copy()

# Only remove rows identical across every selected column.
identical_mask = df_before_exclusions.duplicated()
duplicate_rows = df_before_exclusions.loc[identical_mask].copy()

df_dedup = df_before_exclusions.loc[~identical_mask].copy()

ISSUANCE_KEY = [
    "work_permit", "sequence_number", "issued_date", "borough"
]

no_work = (
    df_dedup["filing_reason"]
    .str.casefold().eq("no work permit")
    .fillna(False)
)

not_issued = df_dedup["work_permit"].str.contains(
    "not yet issued", case=False, na=False
)

missing_key = df_dedup[ISSUANCE_KEY].isna().any(axis=1)

# Flags can overlap, so their counts must not be added together.
review_flags = pd.DataFrame({
    "no_work_permit": no_work,
    "not_yet_issued": not_issued,
    "missing_issuance_key": missing_key
})

exclude_mask = review_flags.any(axis=1)

excluded_records = df_dedup.loc[exclude_mask].copy()
excluded_records = excluded_records.join(
    review_flags.add_prefix("excluded_")
)

df_clean = df_dedup.loc[~exclude_mask].copy()

cleaning_log = pd.DataFrame([
    {
        "step": "Starting records",
        "rows": len(df_before_exclusions)
    },
    {
        "step": "Identical selected-column rows removed",
        "rows": len(duplicate_rows)
    },
    {
        "step": "Ambiguous records set aside (unique rows)",
        "rows": len(excluded_records)
    },
    {
        "step": "Eligible detail rows retained",
        "rows": len(df_clean)
    }
])

print("CLEANING LOG")
display(cleaning_log)

print("\nEXCLUSION FLAGS — COUNTS MAY OVERLAP")
display(review_flags.sum().rename("rows").to_frame())

print(
    "\nDistinct eligible issuances:",
    f"{len(df_clean.drop_duplicates(subset=ISSUANCE_KEY)):,}"
)

# Save the audit evidence alongside the raw data.
duplicate_rows.to_csv(
    OUTPUT_DIR / "building_identical_rows_removed.csv", index=False
)
excluded_records.to_csv(
    OUTPUT_DIR / "building_excluded_records.csv", index=False
)
cleaning_log.to_csv(
    OUTPUT_DIR / "building_cleaning_log.csv", index=False
)

assert (
    len(df_before_exclusions)
    == len(duplicate_rows) + len(excluded_records) + len(df_clean)
)

CLEANING LOG


,step,rows
0,Starting records,484414
1,Identical selected-column rows removed,25
2,Ambiguous records set aside (unique rows),3200
3,Eligible detail rows retained,481189



EXCLUSION FLAGS — COUNTS MAY OVERLAP


,rows
no_work_permit,3183
not_yet_issued,10
missing_issuance_key,14



Distinct eligible issuances: 408,566


In [ ]:
# This cell checks numeric distributions and flags potential data problems.
# IQR identifies unusually high costs, but an outlier is not automatically an error.
# Date and coordinate checks identify records needing caution.
# We retain records at this stage because these fields do not determine permit counts.

print("NUMERIC SUMMARY")
display(
    df_clean[["estimated_job_costs", "latitude", "longitude"]]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
    .round(2)
)

# Use non-negative costs to calculate the upper IQR threshold.
valid_costs = df_clean.loc[
    df_clean["estimated_job_costs"] >= 0,
    "estimated_job_costs"
]

q1 = valid_costs.quantile(0.25)
q3 = valid_costs.quantile(0.75)
iqr = q3 - q1
upper_cost_limit = q3 + 1.5 * iqr

df_clean["flag_negative_cost"] = (
    df_clean["estimated_job_costs"].lt(0).fillna(False)
)
df_clean["flag_high_cost_iqr"] = (
    df_clean["estimated_job_costs"].gt(upper_cost_limit).fillna(False)
)

# A broad NYC-area box is a screening rule, not an exact city boundary.
coordinates_present = df_clean[
    ["latitude", "longitude"]
].notna().all(axis=1)

inside_screening_box = (
    df_clean["latitude"].between(40.4, 41.0)
    & df_clean["longitude"].between(-74.3, -73.6)
).fillna(False)

df_clean["flag_coordinates_outside_box"] = (
    coordinates_present & ~inside_screening_box
)

df_clean["flag_approved_after_issued"] = (
    df_clean["approved_date"] > df_clean["issued_date"]
).fillna(False)

# Compare calendar dates so time-of-day differences do not affect the check.
df_clean["flag_expired_before_issued"] = (
    df_clean["expired_date"].dt.normalize()
    < df_clean["issued_date"].dt.normalize()
).fillna(False)

issue_counts = pd.Series({
    "Missing cost": df_clean["estimated_job_costs"].isna().sum(),
    "Zero cost": df_clean["estimated_job_costs"].eq(0).sum(),
    "Negative cost": df_clean["flag_negative_cost"].sum(),
    "High cost above upper IQR threshold": df_clean["flag_high_cost_iqr"].sum(),
    "Missing one or both coordinates": (~coordinates_present).sum(),
    "Coordinates outside screening box":
        df_clean["flag_coordinates_outside_box"].sum(),
    "Approval after issuance": df_clean["flag_approved_after_issued"].sum(),
    "Expiry before issuance": df_clean["flag_expired_before_issued"].sum()
}, name="detail_rows")

print(f"\nUpper cost IQR threshold: ${upper_cost_limit:,.2f}")
display(issue_counts.to_frame())

print("\nTEN HIGHEST-COST DETAIL ROWS")
display(
    df_clean.nlargest(10, "estimated_job_costs")[
        ISSUANCE_KEY + ["work_type", "estimated_job_costs"]
    ]
)

NUMERIC SUMMARY


,estimated_job_costs,latitude,longitude
count,481189.0,478916.0,478916.0
mean,326004.49,40.73,-73.94
std,3961659.24,0.07,0.09
min,0.0,40.5,-74.25
25%,5000.0,40.68,-73.99
50%,26400.0,40.73,-73.96
75%,120000.0,40.77,-73.9
95%,990700.0,40.85,-73.78
99%,4690609.0,40.88,-73.73
max,800010000.0,40.91,-73.7



Upper cost IQR threshold: $292,500.00


,detail_rows
Missing cost,0
Zero cost,33202
Negative cost,0
High cost above upper IQR threshold,71482
Missing one or both coordinates,2273
Coordinates outside screening box,0
Approval after issuance,9
Expiry before issuance,250



TEN HIGHEST-COST DETAIL ROWS


,work_permit,sequence_number,issued_date,borough,work_type,estimated_job_costs
360827,S00549493-I1-MS,3,2025-10-16,STATEN ISLAND,Mechanical Systems,800010000.0
360893,S00549493-I1-MS,2,2025-06-23,STATEN ISLAND,Mechanical Systems,800010000.0
361089,S00549493-I1-MS,1,2024-01-12,STATEN ISLAND,Mechanical Systems,800010000.0
382214,M00615380-I1-GC,1,2025-10-02,MANHATTAN,General Construction,654000000.0
478307,M00615380-I1-GC,2,2025-12-03,MANHATTAN,General Construction,654000000.0
463036,B00944006-I1-SE,3,2024-12-17,BROOKLYN,Support of Excavation,625000000.0
463119,B00944006-I1-SE,1,2024-01-22,BROOKLYN,Support of Excavation,625000000.0
463163,B00944006-I1-SE,2,2024-12-03,BROOKLYN,Support of Excavation,625000000.0
463433,B00944006-I1-SE,4,2025-12-18,BROOKLYN,Support of Excavation,625000000.0
263696,X00885060-I1-GC,1,2023-07-31,BRONX,General Construction,369608898.0


In [ ]:
# This cell creates monthly permit counts for integration with transportation data.
# Overall totals count each distinct issuance once, preserving separate renewals.
# Work-type totals count an issuance once per associated category.
# Category totals overlap, so they must not be summed to obtain overall totals.
# Missing borough-month combinations remain missing until coverage is confirmed.

df_clean["year"] = df_clean["issued_date"].dt.year
df_clean["month"] = df_clean["issued_date"].dt.month
df_clean["month_start"] = (
    df_clean["issued_date"].dt.to_period("M").dt.to_timestamp()
)

# Keep only the fields defining an issuance; do not arbitrarily select
# one work type or cost from an issuance with multiple detail rows.
issuances = df_clean[ISSUANCE_KEY].drop_duplicates().copy()

issuances["year"] = issuances["issued_date"].dt.year
issuances["month"] = issuances["issued_date"].dt.month
issuances["month_start"] = (
    issuances["issued_date"].dt.to_period("M").dt.to_timestamp()
)

monthly_counts = (
    issuances.groupby(
        ["borough", "year", "month", "month_start"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "building_permit_count"})
    .sort_values(["borough", "year", "month"])
    .reset_index(drop=True)
)

# Preserve each distinct issuance-to-work-type association.
issuance_work_types = df_clean[
    ISSUANCE_KEY + ["work_type"]
].drop_duplicates().copy()

issuance_work_types["year"] = (
    issuance_work_types["issued_date"].dt.year
)
issuance_work_types["month"] = (
    issuance_work_types["issued_date"].dt.month
)

monthly_by_work_type = (
    issuance_work_types.groupby(
        ["borough", "year", "month", "work_type"],
        as_index=False,
        dropna=False
    )
    .size()
    .rename(columns={"size": "building_permit_count"})
    .sort_values(["borough", "year", "month", "work_type"])
    .reset_index(drop=True)
)

# Check whether all 5 boroughs have records in all 36 months.
expected_groups = pd.MultiIndex.from_product(
    [
        sorted(valid_boroughs),
        [2023, 2024, 2025],
        range(1, 13)
    ],
    names=["borough", "year", "month"]
)

observed_groups = pd.MultiIndex.from_frame(
    monthly_counts[["borough", "year", "month"]]
)
missing_groups = expected_groups.difference(observed_groups)

assert monthly_counts["building_permit_count"].sum() == len(issuances)
assert not monthly_counts.duplicated(
    ["borough", "year", "month"]
).any()

print(f"Distinct issuances: {len(issuances):,}")
print(f"Monthly groups: {len(monthly_counts)} / 180")
print(f"Missing borough-month groups: {len(missing_groups)}")
print(f"Work-type summary rows: {len(monthly_by_work_type):,}")

if len(missing_groups):
    display(missing_groups.to_frame(index=False))

print("\nMONTHLY COUNTS PREVIEW")
display(monthly_counts.head(10))

print("\nTOTAL DISTINCT ISSUANCES BY BOROUGH")
display(
    monthly_counts.groupby("borough")["building_permit_count"]
    .sum()
    .sort_values(ascending=False)
    .to_frame()
)

# Save cleaned detail, distinct issuances and both monthly summaries.
df_clean.to_csv(
    OUTPUT_DIR / "building_permits_cleaned_2023_2025.csv",
    index=False
)
issuances.to_csv(
    OUTPUT_DIR / "building_issuances_2023_2025.csv",
    index=False
)
monthly_counts.to_csv(
    OUTPUT_DIR / "building_monthly_counts.csv",
    index=False
)
monthly_by_work_type.to_csv(
    OUTPUT_DIR / "building_monthly_by_work_type.csv",
    index=False
)

print("\nCleaned data and monthly summaries saved.")

Distinct issuances: 408,566
Monthly groups: 180 / 180
Missing borough-month groups: 0
Work-type summary rows: 3,468

MONTHLY COUNTS PREVIEW


,borough,year,month,month_start,building_permit_count
0,BRONX,2023,1,2023-01-01,955
1,BRONX,2023,2,2023-02-01,989
2,BRONX,2023,3,2023-03-01,1341
3,BRONX,2023,4,2023-04-01,1076
4,BRONX,2023,5,2023-05-01,1099
5,BRONX,2023,6,2023-06-01,1105
6,BRONX,2023,7,2023-07-01,956
7,BRONX,2023,8,2023-08-01,1030
8,BRONX,2023,9,2023-09-01,991
9,BRONX,2023,10,2023-10-01,973



TOTAL DISTINCT ISSUANCES BY BOROUGH


,building_permit_count
borough,
MANHATTAN,160885
BROOKLYN,109394
QUEENS,75985
BRONX,41495
STATEN ISLAND,20807



Cleaned data and monthly summaries saved.


In [ ]:
# This cell visualizes the cleaned data to compare permit activity across boroughs
# and inspect monthly patterns. Counts represent distinct permit issuances,
# including renewals, rather than unique buildings or completed construction.
# These EDA charts describe housing/development data before cross-domain analysis.

borough_totals = (
    monthly_counts.groupby("borough", as_index=False)
    ["building_permit_count"]
    .sum()
    .sort_values("building_permit_count", ascending=False)
)

fig_borough = px.bar(
    borough_totals,
    x="borough",
    y="building_permit_count",
    color="borough",
    text="building_permit_count",
    title="Distinct Building Permit Issuances by Borough, 2023–2025",
    labels={
        "borough": "Borough",
        "building_permit_count": "Distinct permit issuances"
    },
    template="plotly_white"
)

fig_borough.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)
fig_borough.update_layout(showlegend=False)
fig_borough.show()

fig_monthly = px.line(
    monthly_counts.sort_values(["borough", "month_start"]),
    x="month_start",
    y="building_permit_count",
    color="borough",
    markers=True,
    title="Monthly Building Permit Issuances by Borough, 2023–2025",
    labels={
        "month_start": "Issuance month",
        "building_permit_count": "Distinct permit issuances",
        "borough": "Borough"
    },
    template="plotly_white"
)

fig_monthly.update_layout(hovermode="x unified")
fig_monthly.show()

# Save standalone interactive copies that open in a browser.
fig_borough.write_html(
    OUTPUT_DIR / "eda_permits_by_borough.html",
    include_plotlyjs=True
)
fig_monthly.write_html(
    OUTPUT_DIR / "eda_monthly_permit_trends.html",
    include_plotlyjs=True
)

print("Both EDA charts saved as interactive HTML files.")

# Print the peak observed month for each borough.
peak_indices = monthly_counts.groupby(
    "borough"
)["building_permit_count"].idxmax()

print("\nPEAK MONTH BY BOROUGH")
display(
    monthly_counts.loc[
        peak_indices,
        ["borough", "month_start", "building_permit_count"]
    ].sort_values("borough").reset_index(drop=True)
)

Both EDA charts saved as interactive HTML files.

PEAK MONTH BY BOROUGH


,borough,month_start,building_permit_count
0,BRONX,2025-07-01,1386
1,BROOKLYN,2025-07-01,3563
2,MANHATTAN,2024-05-01,5103
3,QUEENS,2024-10-01,2464
4,STATEN ISLAND,2025-10-01,783


### EDA findings and limitations

The cleaned dataset contains 408,566 distinct eligible permit issuances
during 2023–2025, including initial permits and renewals. All 180 expected
borough-month combinations contain records.

Manhattan has the highest total, with 160,885 issuances, followed by
Brooklyn (109,394), Queens (75,985), the Bronx (41,495), and Staten Island
(20,807). These are absolute counts, not rates adjusted for population,
building stock or borough size.

The highest observed month differs between boroughs:
- Bronx: July 2025 — 1,386 issuances.
- Brooklyn: July 2025 — 3,563 issuances.
- Manhattan: May 2024 — 5,103 issuances.
- Queens: October 2024 — 2,464 issuances.
- Staten Island: October 2025 — 783 issuances.

These peaks describe the observed period and do not establish a recurring
seasonal pattern or explain why activity changed.

An issuance is defined using work permit, sequence number, issued date
and borough. One issuance may cover several work types, so category
counts overlap and must not be summed to obtain overall totals.

The analysis measures recorded permit issuance activity, not unique
buildings, completed construction or new homes. The two EDA charts do
not yet establish a relationship with transportation; that requires
integration with the street construction permit dataset.

### Research questions: building and street permit activity

**Question 1:** Across borough-months during 2023–2025, are higher
building permit issuance counts associated with higher street permit counts?

**Visualization:** An interactive scatter plot with building issuance
counts on the x-axis and street permit counts on the y-axis.
Each point represents one borough-month, colored by borough.
Hover details will identify the borough, year, month and both counts.

**Analysis:** Calculate Pearson correlation using matched observations.
Interpret the pooled result cautiously because differences in borough
size and baseline activity can influence the relationship.

**Question 2:** Does the relationship between building and street
permit activity differ across NYC boroughs?

**Visualization:** Separate scatter panels for each borough, with
building issuance counts on the x-axis and street permit counts
on the y-axis.

**Analysis:** Calculate Pearson correlation separately for each borough
and report the number of matched months. Each borough can contribute
up to 36 observations. Correlation is undefined when either count
has no variation.

**Integration and limitations**
Both questions use data joined by borough, year and month.
Individual building permits are not matched to individual street permits.

The analysis compares recorded permit activity. It does not establish
causation, completed construction, congestion or whether permits belong
to the same project. Shared trends and seasonality can influence
correlations.

Results from the real integrated dataset are reported below.

In [1]:
# This cell defines reusable interactive charts for our two research questions.
# Both functions accept integrated monthly data and return a Plotly figure
# plus a correlation table. Only complete, unique borough-month pairs are used.
# Correlation is left undefined when there are fewer than 3 pairs or no variation.
# Defining the functions does not calculate results before real data is available.

import pandas as pd
import plotly.express as px

def prepare_comparison(data):
    keys = ["borough", "year", "month"]
    metrics = ["building_permit_count", "street_permit_count"]
    required = keys + metrics

    missing = set(required) - set(data.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    comparison = data[required].copy()

    if comparison[keys].isna().any().any():
        raise ValueError("Missing borough, year or month values.")

    if comparison.duplicated(keys).any():
        raise ValueError("Expected one row per borough/year/month.")

    for column in metrics:
        comparison[column] = pd.to_numeric(
            comparison[column], errors="raise"
        )
        observed = comparison[column].dropna()

        if not observed.between(0, float("inf"), inclusive="left").all():
            raise ValueError(f"Invalid counts in {column}.")

    # Unmatched counts remain unknown, rather than being replaced with zero.
    comparison = comparison.dropna(subset=metrics)

    if comparison.empty:
        raise ValueError("No complete matched observations.")

    return comparison.sort_values(keys)


def correlation_summary(group, label):
    n = len(group)
    varies = (
        group["building_permit_count"].nunique() > 1
        and group["street_permit_count"].nunique() > 1
    )

    r = (
        group["building_permit_count"].corr(
            group["street_permit_count"]
        )
        if n >= 3 and varies
        else float("nan")
    )

    return {
        "group": label,
        "matched_months": n,
        "pearson_r": r
    }


def make_q1_chart(data):
    comparison = prepare_comparison(data)

    figure = px.scatter(
        comparison,
        x="building_permit_count",
        y="street_permit_count",
        color="borough",
        hover_data=["year", "month"],
        title="Q1: Building and Street Permit Activity",
        labels={
            "building_permit_count": "Building permit issuances",
            "street_permit_count": "Street permit count",
            "borough": "Borough"
        },
        template="plotly_white"
    )

    summary = pd.DataFrame([
        correlation_summary(comparison, "All borough-months")
    ])

    return figure, summary


def make_q2_chart(data):
    comparison = prepare_comparison(data)

    figure = px.scatter(
        comparison,
        x="building_permit_count",
        y="street_permit_count",
        color="borough",
        facet_col="borough",
        facet_col_wrap=3,
        hover_data=["year", "month"],
        title="Q2: Building–Street Permit Relationship by Borough",
        labels={
            "building_permit_count": "Building permit issuances",
            "street_permit_count": "Street permit count",
            "borough": "Borough"
        },
        template="plotly_white",
        height=700
    )

    # Shared axes allow direct comparison of activity levels across boroughs.
    figure.update_layout(showlegend=False)
    figure.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.split("=")[-1]
        )
    )

    summary = pd.DataFrame([
        correlation_summary(group, borough)
        for borough, group in comparison.groupby("borough")
    ])

    return figure, summary


print("Q1 and Q2 chart functions are ready.")

Q1 and Q2 chart functions are ready.


In [ ]:
# Load the actual integrated result and create the two assigned research charts.
task1_integrated = load_saved_csv('integrated_monthly_permits.csv')
assert len(task1_integrated) == 180
assert task1_integrated['_merge'].eq('both').all()
assert task1_integrated['building_permit_count'].sum() == 408566
assert task1_integrated['street_permit_count'].sum() == 1951371
fig_q1, q1_summary = make_q1_chart(task1_integrated)
fig_q2, q2_summary = make_q2_chart(task1_integrated)
fig_q1.show(); display(q1_summary)
fig_q2.show(); display(q2_summary)
for fig, name in [(fig_q1,'q1_overall_relationship'),(fig_q2,'q2_relationship_by_borough')]:
    fig.write_html(str(OUTPUT_DIR / (name+'.html')), include_plotlyjs=True)
q1_summary.to_csv(OUTPUT_DIR/'q1_correlation.csv',index=False)
q2_summary.to_csv(OUTPUT_DIR/'q2_borough_correlations.csv',index=False)

### Research findings

#### Question 1: Are building and street permit counts associated?

Across 180 matched borough-month observations during 2023–2025,
the Pearson correlation is 0.8230, indicating a strong positive
pooled linear association.

However, this combines differences between boroughs with monthly
variation within boroughs. It does not show that an increase in
building permits within a borough necessarily corresponds to a
similarly strong increase in street permits.

#### Question 2: Does the relationship differ between boroughs?

Each borough contributes 36 matched monthly observations.

| Borough | Pearson correlation |
|---|---:|
| Queens | 0.4477 |
| Brooklyn | 0.4470 |
| Manhattan | 0.3467 |
| Bronx | 0.1815 |
| Staten Island | -0.0468 |

Queens and Brooklyn show moderate positive associations, Manhattan
a smaller positive association, and the Bronx a weak positive
association. Staten Island shows little contemporaneous linear
association.

All borough-specific correlations are substantially smaller than
the pooled correlation. This suggests that differences in baseline
activity between boroughs contribute to the strong pooled result.
The pooled value should not be presented as evidence of strong
month-to-month alignment within every borough.

#### Limitations and implications

The analysis compares permit activity within the same borough-month,
not individual permits belonging to the same project.

Different counting rules, renewals, shared seasonality and trends
can influence the relationships. Monthly observations may be serially
related. No statistical significance or causal conclusions are claimed.

Permit counts do not directly measure completed construction,
new homes, congestion or infrastructure capacity.

The findings support examining borough-specific patterns rather than
using one citywide relationship to describe all boroughs. Further
investigation could assess whether differences reflect project types,
administrative processes or timing between activities.

### AI assistance and prompt log

ChatGPT assisted with planning, Python code, interpretation of diagnostic
outputs, and documentation. The code was executed in Google Colab,
and cleaning decisions were reviewed using the resulting outputs.

The following excerpts record prompts used during this work.
Follow-up messages also supplied execution results for review.

| Stage | Prompt excerpt | Assistance received |
|---|---|---|
| Project planning | "analyse it very thoroughly, then explain everything, distrubute the work over 4 prople" | Requirements analysis and initial task distribution; subsequently revised to five members |
| Topic selection | "what do you think is the fastest topic we could go for, choose for us" | Selection of building and street construction permits and a proposed analysis scope |
| Task 1 implementation | "im handling task 1, i need you to walk me through step by step" | Data loading, EDA, cleaning, validation and export code |
| Code documentation | "after every code block can you send me a small comment to add at the top?" | Explanatory comments for notebook cells |
| Team review | "analyse the 3 things with the project description along with the task list" | Review of submitted files, identification of synthetic street data and incomplete integration |
| Research preparation | "is there anything in the meantime to do in the other 2 tasks?" | Research-question documentation and reusable chart functions |

Both research charts were executed using the real integrated dataset. Findings were reviewed against the resulting correlation tables.

### Handoff instructions for integration and website development

**Scope:** January 2023–December 2025, all five NYC boroughs.

#### Files

- building_monthly_counts.csv
  Overall distinct issuance counts.
  One row per borough + year + month.
  Contains 180 rows, totaling 408,566 issuances.

- building_monthly_by_work_type.csv
  Distinct issuance counts within each work type.
  One row per borough + year + month + work_type.
  Contains 3,468 rows. Categories overlap.

- building_issuances_2023_2025.csv
  One row per distinct analytical issuance.

- building_permits_cleaned_2023_2025.csv
  Cleaned work-type detail records, including quality flags.
  Contains 481,189 rows.

#### Integration rules

Join the overall monthly table to the transportation monthly table using:
borough + year + month.

Standardize transportation borough labels to:
BRONX, BROOKLYN, MANHATTAN, QUEENS, STATEN ISLAND.

Both tables must have unique borough/year/month keys before joining.
Use a one-to-one merge validation and inspect unmatched groups.

The building count column is:
building_permit_count.

Do not replace unmatched transportation values with zero unless
transportation coverage confirms that zero is meaningful.

#### Website filtering rules

For "All work types", use building_monthly_counts.csv.

For one selected work type, use building_monthly_by_work_type.csv.

For multiple selected work types, filter the cleaned detail records,
then deduplicate using:
work_permit + sequence_number + issued_date + borough.
Aggregate those distinct issuances by borough/year/month.

Do not sum work-type counts to produce an overall or multi-type total:
one issuance can belong to several work types.

Street permit category filters apply to transportation records.
Building work-type filters apply to building records.
These filters compare activity in the same borough-month;
they do not establish links between individual permits.

#### Transportation input used for the completed research charts

Provide a cleaned transportation summary containing:
borough, year, month, street_permit_count.

The table must contain one row per borough/year/month and include
an explanation of its counting rule and any missing coverage.

Task 1 questions:
1. Do borough-months with more building permit issuances also have
   more street permit issuances?
2. Does that relationship differ between boroughs?